# vLLM Multi-Gemma — Build, Push & Deploy

This notebook:
1. Clones the repo and builds the Docker image
2. Pushes it to GitHub Container Registry (GHCR)
3. Launches a vast.ai instance with the image
4. Tests the endpoint

**Prerequisites:**
- GitHub PAT (classic) with `write:packages` and `read:packages` scopes
- HuggingFace token with access to gated Gemma models
- vast.ai API key (from https://cloud.vast.ai/account/)

---

## 0. Configuration
Fill in your credentials below. These stay in your Colab runtime only.

In [ ]:
# ══════════════════════════════════════════════
#  FILL THESE IN
# ══════════════════════════════════════════════

GITHUB_USER = "your-github-username"        # GitHub username (lowercase)
GITHUB_PAT  = "ghp_xxxxxxxxxxxxxxxxxxxx"     # PAT with write:packages scope
HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxx"      # HuggingFace token
VASTAI_KEY  = "xxxxxxxxxxxxxxxxxxxxxxxx"     # vast.ai API key

# Image settings
IMAGE_NAME  = f"ghcr.io/{GITHUB_USER}/vllm-multi-gemma"
IMAGE_TAG   = "latest"
FULL_IMAGE  = f"{IMAGE_NAME}:{IMAGE_TAG}"

# Model to start with
INITIAL_MODEL = "gemma2-cosmos"  # or gemma3-12b, gemma4-12b

print(f"Image: {FULL_IMAGE}")
print(f"Model: {INITIAL_MODEL}")

## 1. Install Dependencies

In [ ]:
!pip install -q vastai

# Set vast.ai API key
!vastai set api-key {VASTAI_KEY}

print("✓ Dependencies installed")

## 2. Clone Repo & Build Docker Image

In [ ]:
%%bash -s "$GITHUB_USER" "$GITHUB_PAT"

# Clone repo (replace URL with your actual repo)
REPO_URL="https://$1:$2@github.com/$1/vllm-multi-gemma.git"

if [ -d "vllm-multi-gemma" ]; then
    echo "Repo already cloned, pulling latest..."
    cd vllm-multi-gemma && git pull
else
    git clone "$REPO_URL"
    cd vllm-multi-gemma
fi

echo "✓ Repo ready"
ls -la

In [ ]:
%%bash -s "$GITHUB_USER" "$GITHUB_PAT" "$FULL_IMAGE"

# Login to GHCR
echo "$2" | docker login ghcr.io -u "$1" --password-stdin

# Build image
cd vllm-multi-gemma
echo "Building Docker image: $3"
docker build -t "$3" -f docker/Dockerfile .

echo ""
echo "✓ Image built: $3"
docker images | grep vllm-multi-gemma

## 3. Push to GHCR

In [ ]:
%%bash -s "$FULL_IMAGE"

echo "Pushing $1 to GHCR..."
docker push "$1"

echo ""
echo "✓ Image pushed: $1"
echo "Make sure the package is public at: https://github.com/users/$(echo $1 | cut -d/ -f2)/packages/container/vllm-multi-gemma"

### ⚠️ Make the GHCR package public

vast.ai needs to pull without auth. Go to:

`GitHub → Your Profile → Packages → vllm-multi-gemma → Package Settings → Danger Zone → Change visibility → Public`

Or keep it private and configure vast.ai with Docker registry credentials.

## 4. Find & Launch vast.ai Instance

In [ ]:
# Search for suitable GPU instances
!vastai search offers \
    'gpu_ram>=32 num_gpus=1 reliability>0.95 inet_down>200 cuda_vers>=12.0 disk_space>=100' \
    -o 'dph_total' \
    --limit 10

In [ ]:
# ══════════════════════════════════════════════
#  Pick an instance ID from the search results above
# ══════════════════════════════════════════════

OFFER_ID = 12345678  # <--- REPLACE with actual offer ID from search results

In [ ]:
import subprocess, json

# Launch the instance
env_str = (
    f"-e MODEL_NAME={INITIAL_MODEL} "
    f"-e HF_TOKEN={HF_TOKEN} "
    f"-e PORT=8000 "
    f"-e MAX_MODEL_LEN=4096 "
    f"-e GPU_MEMORY_UTILIZATION=0.92 "
    f"-e DTYPE=bfloat16"
)

cmd = (
    f"vastai create instance {OFFER_ID} "
    f"--image {FULL_IMAGE} "
    f"--disk 100 "
    f"--env '{env_str}' "
    f"--direct "
    f"--onstart-cmd '' "
)

print(f"Launching: {cmd}")
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(f"stderr: {result.stderr}")

# Extract instance ID from output
# vast.ai outputs something like: "Started. {'new_contract': 12345678}"
print("\n→ Copy the instance ID and set it in the next cell")

In [ ]:
# ══════════════════════════════════════════════
#  Set your instance ID
# ══════════════════════════════════════════════

INSTANCE_ID = 12345678  # <--- REPLACE with actual instance ID

In [ ]:
# Monitor instance status
!vastai show instance {INSTANCE_ID} --raw | python3 -c "
import sys, json
d = json.load(sys.stdin)
print(f\"Status:     {d.get('actual_status', 'unknown')}\")
print(f\"GPU:        {d.get('gpu_name', '?')} x{d.get('num_gpus', '?')}\")
print(f\"SSH:        ssh -p {d.get('ssh_port', '?')} root@{d.get('ssh_host', '?')}\")
port_map = d.get('ports', {})
if '8000/tcp' in port_map:
    host_info = port_map['8000/tcp'][0]
    print(f\"Endpoint:   http://{host_info['HostIp']}:{host_info['HostPort']}/v1\")
else:
    print(f\"Endpoint:   Waiting for port mapping...\")
    print(f\"  Direct:   https://{INSTANCE_ID}-8000.proxy.vast.ai/v1\")
"

## 5. Wait for Model to Load & Test

In [ ]:
import time, urllib.request, json

# Determine endpoint URL
# vast.ai proxy format — adjust if using direct port mapping
BASE_URL = f"https://{INSTANCE_ID}-8000.proxy.vast.ai"

print(f"Waiting for vLLM at {BASE_URL} ...")
print("(Model download + load can take 5-15 minutes)\n")

for attempt in range(60):  # 10 min max
    try:
        req = urllib.request.Request(f"{BASE_URL}/health")
        urllib.request.urlopen(req, timeout=5)
        print(f"\n✓ Server is ready! (attempt {attempt + 1})")
        break
    except Exception:
        print(f"  Attempt {attempt + 1}/60 — not ready yet...", end="\r")
        time.sleep(10)
else:
    print("\n✗ Server did not become ready. Check logs:")
    print(f"  vastai logs {INSTANCE_ID}")

In [ ]:
# List loaded models
req = urllib.request.Request(
    f"{BASE_URL}/v1/models",
    headers={"Authorization": "Bearer not-needed"}
)
resp = urllib.request.urlopen(req, timeout=10)
data = json.loads(resp.read())

for m in data["data"]:
    print(f"Model: {m['id']}")

In [ ]:
# Test chat completion
model_id = data["data"][0]["id"]

payload = json.dumps({
    "model": model_id,
    "messages": [
        {"role": "user", "content": "Merhaba! Kendini kısaca tanıt."}
    ],
    "max_tokens": 256,
    "temperature": 0.7,
}).encode()

req = urllib.request.Request(
    f"{BASE_URL}/v1/chat/completions",
    data=payload,
    headers={
        "Content-Type": "application/json",
        "Authorization": "Bearer not-needed",
    },
    method="POST",
)

resp = urllib.request.urlopen(req, timeout=60)
result = json.loads(resp.read())

reply = result["choices"][0]["message"]["content"]
usage = result["usage"]

print(f"Model: {model_id}")
print(f"Reply: {reply}")
print(f"\nTokens: prompt={usage['prompt_tokens']}, completion={usage['completion_tokens']}")

## 6. Switch Model (via SSH)

To switch the running model without restarting the container:

```bash
# SSH into the instance
ssh -p <PORT> root@<HOST>

# Switch model
/opt/vllm-gemma/scripts/switch_model.sh gemma3-12b

# Or with full HF repo ID
/opt/vllm-gemma/scripts/switch_model.sh google/gemma-4-12b-it

# Check status
/opt/vllm-gemma/scripts/health_check.sh
```

## 7. Cleanup

In [ ]:
# Destroy the instance when done
# !vastai destroy instance {INSTANCE_ID}
# print(f"Instance {INSTANCE_ID} destroyed")

---

## Quick Reference

| Action | Command |
|--------|---------|
| Check status | `vastai show instance INSTANCE_ID` |
| View logs | `vastai logs INSTANCE_ID` |
| SSH in | `ssh -p PORT root@HOST` |
| Switch model | `/opt/vllm-gemma/scripts/switch_model.sh gemma3-12b` |
| Health check | `/opt/vllm-gemma/scripts/health_check.sh` |
| Pre-download all | `/opt/vllm-gemma/scripts/download_models.sh` |
| Destroy | `vastai destroy instance INSTANCE_ID` |

### vast.ai Endpoint URL Formats
- **Proxy**: `https://INSTANCE_ID-8000.proxy.vast.ai/v1`
- **Direct**: `http://HOST:MAPPED_PORT/v1` (from port mapping)

### Model Shortcuts
| Shortcut | HuggingFace Repo |
|----------|------------------|
| `gemma2-cosmos` | `ytu-ce-cosmos/turkish-gemma2-9b-it` |
| `gemma3-12b` | `google/gemma-3-12b-it` |
| `gemma4-12b` | `google/gemma-4-12b-it` |